# Evaluation and Monitoring

Evaluating AI agents requires more than exact-match metrics. The Evaluation and Monitoring pattern combines: (1) token usage tracking for cost control, (2) latency monitoring for SLA compliance, and (3) LLM-as-a-Judge for subjective quality dimensions that rule-based metrics can't capture.

## Implementation with Flyte v2

This notebook reimplements the `LLMJudgeForLegalSurvey` + `LLMInteractionMonitor` pattern from Chapter 19 using **Flyte v2 primitives + Anthropic API** — replacing Gemini with Anthropic.

#### Original (Gemini) vs Flyte v2 (Anthropic) — Key Differences

| Aspect | Original (Gemini) | Flyte v2 |
|--------|-------------------|----------|
| **LLM client** | `genai.GenerativeModel` at module level | `AsyncAnthropic` inside task — safe in containers |
| **JSON output** | `response_mime_type="application/json"` | Prompt-enforced JSON + Pydantic `model_validate()` |
| **Token tracking** | `len(text.split())` placeholder | Anthropic `response.usage.input_tokens / output_tokens` |
| **Caching** | None (re-evaluates identical inputs) | `cache="auto"` — skips redundant LLM evaluations |
| **Secrets** | `os.environ["GOOGLE_API_KEY"]` at import time | `flyte.Secret` injected at task runtime |
| **Observability** | `logging` + `print()` | Structured `EvaluationReport` in Flyte UI |
| **Execution** | In-process only | Local or remote (containers) |

> **🧭 When to use this pattern — and how Flyte helps**
>
> Use evaluation and monitoring for agents in live production where reliability, quality, and cost must be tracked continuously. Flyte fits naturally: an evaluation run is a deterministic `@env.task` + `flyte.map` pipeline with cached, reproducible scoring, and per-task metrics, reports, and traces surface latency and quality without log scraping.

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' anthropic pydantic

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...

### 3. Import dependencies and configure the TaskEnvironment

In [1]:

import os
import time
from dataclasses import dataclass, field
from datetime import timedelta
from typing import List

from anthropic import AsyncAnthropic
import flyte
from pydantic import BaseModel, Field

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="eval-agent", python_version=(3, 12))
    .with_pip_packages("anthropic>=0.25.0", "pydantic>=2.0.0")
)

eval_env = flyte.TaskEnvironment(
    name="eval_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="1Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
)

### 4. Define the LLM-as-a-Judge rubric and output schema

The original `LLMJudgeForLegalSurvey` used a Gemini model with `response_mime_type="application/json"` to enforce structured output. In Flyte v2, `JudgmentResult` is a Pydantic model validated with `model_validate()` — the same guarantees without the Gemini-specific parameter.

The rubric (`LEGAL_SURVEY_RUBRIC`) is preserved verbatim from the original, with the model swapped from Gemini to Anthropic.

In [3]:
LEGAL_SURVEY_RUBRIC = """\
You are an expert legal survey methodologist. Evaluate the quality of a legal survey question.
Score each criterion 1-5 and return ONLY valid JSON.

Criteria:
1. Clarity & Precision (1=vague, 5=perfectly clear)
2. Neutrality & Bias (1=highly leading, 5=completely neutral)
3. Relevance & Focus (1=irrelevant, 5=directly on topic)
4. Completeness (1=missing critical info, 5=fully contextualized)
5. Audience Appropriateness (1=wrong level, 5=perfectly tailored)

Response JSON schema (return ONLY this, no markdown fences):
{
  "overall_score": <int 1-5>,
  "clarity_score": <int 1-5>,
  "neutrality_score": <int 1-5>,
  "relevance_score": <int 1-5>,
  "completeness_score": <int 1-5>,
  "audience_score": <int 1-5>,
  "rationale": "<brief summary>",
  "detailed_feedback": ["<per-criterion feedback>"],
  "concerns": ["<legal/ethical concern>"],
  "recommended_action": "<Approve as is | Revise for neutrality | Clarify scope | Reject>"
}"""


class JudgmentResult(BaseModel):
    """
    Structured output from the LLM judge.
    Replaces the raw dict returned by LLMJudgeForLegalSurvey.judge_survey_question().
    """
    overall_score: int = Field(ge=1, le=5)
    clarity_score: int = Field(ge=1, le=5)
    neutrality_score: int = Field(ge=1, le=5)
    relevance_score: int = Field(ge=1, le=5)
    completeness_score: int = Field(ge=1, le=5)
    audience_score: int = Field(ge=1, le=5)
    rationale: str
    detailed_feedback: List[str]
    concerns: List[str]
    recommended_action: str


class EvaluationReport(BaseModel):
    """Full evaluation report including latency and token tracking."""
    survey_question: str
    judgment: JudgmentResult
    latency_ms: float
    input_tokens: int
    output_tokens: int
    total_tokens: int

### 5. Define the LLM Interaction Monitor

**Why `cache="auto"` here?** For repeated evaluations of the same question (e.g., regression tests), the LLM judge produces identical output at `temperature=0`. Caching eliminates redundant API calls — 100 runs of the same question cost the same as 1.

In [4]:
@eval_env.task(cache="auto", retries=2, timeout=timedelta(minutes=3))
async def judge_survey_question(survey_question: str) -> EvaluationReport:
    """
    LLM-as-a-Judge evaluation of a legal survey question.

    Replaces LLMJudgeForLegalSurvey.judge_survey_question() which used:
      genai.GenerativeModel(model_name)
      model.generate_content(prompt, generation_config=GenerationConfig(
          temperature=0.2,
          response_mime_type='application/json'
      ))

    Key improvements:
    - Exact token counts from response.usage (not split() approximation)
    - Pydantic validation instead of raw json.loads + dict
    - cache='auto' eliminates redundant evaluations for identical inputs
    """
    import json

    client = AsyncAnthropic()
    start = time.monotonic()

    response = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        temperature=0.2,
        system=LEGAL_SURVEY_RUBRIC,
        messages=[{
            "role": "user",
            "content": f"Legal survey question to evaluate:\n\n{survey_question}",
        }],
    )

    latency_ms = (time.monotonic() - start) * 1000

    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

    judgment = JudgmentResult.model_validate(json.loads(raw))

    # Exact token tracking from Anthropic API — replaces len(text.split()) placeholder
    return EvaluationReport(
        survey_question=survey_question,
        judgment=judgment,
        latency_ms=latency_ms,
        input_tokens=response.usage.input_tokens,
        output_tokens=response.usage.output_tokens,
        total_tokens=response.usage.input_tokens + response.usage.output_tokens,
    )

### 6. Define the batch evaluation task

The original code evaluated questions sequentially. In Flyte v2, `asyncio.gather` parallelizes evaluations — multiple questions are judged concurrently on the same warm pod.

In [5]:
class BatchEvaluationSummary(BaseModel):
    """Aggregated metrics from a batch evaluation run."""
    questions_evaluated: int
    average_overall_score: float
    approved_count: int
    revision_needed_count: int
    rejected_count: int
    total_tokens_used: int
    average_latency_ms: float
    reports: List[EvaluationReport]


@eval_env.task(
    retries=1,
    timeout=timedelta(minutes=10),
    cache=flyte.Cache(behavior="disable"),
)
async def batch_evaluate(
    survey_questions: list[str],
) -> BatchEvaluationSummary:
    """Evaluate multiple survey questions in parallel."""
    import asyncio

    reports = list(await asyncio.gather(*[
        judge_survey_question(survey_question=q)
        for q in survey_questions
    ]))

    approved = sum(1 for r in reports if "approve" in r.judgment.recommended_action.lower())
    revision = sum(1 for r in reports if "revise" in r.judgment.recommended_action.lower())
    rejected = sum(1 for r in reports if "reject" in r.judgment.recommended_action.lower())

    return BatchEvaluationSummary(
        questions_evaluated=len(reports),
        average_overall_score=sum(r.judgment.overall_score for r in reports) / len(reports),
        approved_count=approved,
        revision_needed_count=revision,
        rejected_count=rejected,
        total_tokens_used=sum(r.total_tokens for r in reports),
        average_latency_ms=sum(r.latency_ms for r in reports) / len(reports),
        reports=reports,
    )

### 7. Run on the devbox

In [6]:
SURVEY_QUESTIONS = [
    # Good example from original
    """To what extent do you agree or disagree that current intellectual property laws """
    """in Switzerland adequately protect emerging AI-generated content, assuming the """
    """content meets the originality criteria established by the Federal Supreme Court? """
    """(Select one: Strongly Disagree, Disagree, Neutral, Agree, Strongly Agree)""",

    # Biased example from original
    """Don't you agree that overly restrictive data privacy laws like the FADP are """
    """hindering essential technological innovation and economic growth in Switzerland? """
    """(Select one: Yes, No)""",

    # Additional test: vague question
    """What do you think about AI law?""",
]

run = flyte.run(batch_evaluate, survey_questions=SURVEY_QUESTIONS)
run.wait()
summary: BatchEvaluationSummary = run.outputs()[0]

print(f"Questions evaluated: {summary.questions_evaluated}")
print(f"Average score: {summary.average_overall_score:.2f}/5")
print(f"Approved: {summary.approved_count} | Revision: {summary.revision_needed_count} | Rejected: {summary.rejected_count}")
print(f"Total tokens: {summary.total_tokens_used} | Avg latency: {summary.average_latency_ms:.0f}ms")
print()

for report in summary.reports:
    q_short = report.survey_question[:70].replace("\n", " ")
    j = report.judgment
    print(f"Score {j.overall_score}/5 | {j.recommended_action} | {q_short}...")
    print(f"  Rationale: {j.rationale[:100]}")
    print(f"  Tokens: {report.input_tokens}in / {report.output_tokens}out | Latency: {report.latency_ms:.0f}ms")
    print()

> Building 1 image...

> Building image eval-agent for environment eval_agent

i Image localhost:30000/eval-agent:6e5ffb6b434f24a301b355b5aff6ad29 already exists, skipping build

✓ Built image for environment eval_agent: localhost:30000/eval-agent:6e5ffb6b434f24a301b355b5aff6ad29

Output()

Questions evaluated: 3
Average score: 1.67/5
Approved: 0 | Revision: 1 | Rejected: 2
Total tokens: 2567 | Avg latency: 5745ms

Score 3/5 | Revise for neutrality | To what extent do you agree or disagree that current intellectual prop...
  Rationale: The question addresses a relevant legal topic with appropriate audience targeting, but contains stru
  Tokens: 397in / 499out | Latency: 5964ms

Score 1/5 | Reject | Don't you agree that overly restrictive data privacy laws like the FAD...
  Rationale: This question is fundamentally flawed due to severe leading bias. It contains multiple loaded assump
  Tokens: 369in / 507out | Latency: 6434ms

Score 1/5 | Reject | What do you think about AI law?...
  Rationale: This question is fundamentally flawed as a legal survey instrument. It lacks specificity, fails to d
  Tokens: 340in / 455out | Latency: 4838ms



## Why not the Agent harness here?

Evaluation is a **deterministic pipeline**: run the system over a dataset, score each output, aggregate the metrics. Every step has a fixed input/output contract that `@env.task` + `flyte.map` model directly and reproducibly. Here the `Agent` is the *subject* being evaluated, not the right tool to *implement* the evaluation — wrapping the harness around scoring would only make results less reproducible.